# 📘 SHAPE Agent 16 — Section 1: Learning Design
## Authored by: **Cherluk**
### ITAI2376 · Section 14562 · Houston Community College

---

| | |
|---|---|
| **Author** | Cherluk |
| **Role** | Learning Design & Instructional Experience |
| **Design Contribution** | Problem definition, curriculum structure, lesson authoring, Lesson Builder UX, Wake-Phase episodic encoding |
| **Coding** | Curriculum seeding functions, lesson validation, content authoring cells, `log_episode()` wake-phase calls |
| **Reflection** | See Section 6 of this notebook |

**Full team:** Michael Rivas · Cherluk · Olu · Taki

---

## What this notebook covers

This is **Section 1 of 4** in the SHAPE Agent 16 development series.  
I defined the problem, designed the curriculum schema, authored the starter lessons  
that the AI tutor draws from, and encoded the first **Wake-Phase episodes** that seed  
the agent's episodic memory (Step 1 of the 4-Step Memory Demo Protocol).

**Handoff:** My output (populated curriculum DB + seeded wake-phase episodes) is consumed by Olu in Section 2.

---

**Filename:** `NB_Cherluk_Section1_LearningDesign_Group_ITAI2376_14562.ipynb`

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────────────
!pip install -q groq gradio python-dotenv

import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/mrivassnj-svg/SHAPE_Agent16.git'
REPO_DIR = '/content/SHAPE_Agent16'
if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --quiet
os.chdir(REPO_DIR)
sys.path.insert(0, 'src')
print('✅ Ready.')

---
## 1. Problem Statement

**Author: Cherluk**

Community college students learning Python and AI face three persistent barriers:

1. **Uneven prior knowledge** — a single lecture pace leaves fast learners bored and slow learners lost.
2. **Delayed feedback** — graded assignments return days later, long after the learning moment has passed.
3. **Static curriculum** — instructors cannot push targeted lessons mid-week without a full deployment pipeline.

**SHAPE Agent 16** solves all three:
- Detects each learner's level and adjusts its explanation style in real time
- Returns instant feedback on code submissions and answers
- Provides a no-code web form so any team member can publish new lessons immediately

**Domain:** Programming education (Python + AI concepts) — directly aligned with ITAI2376 course material.

**Project Option:** Option 4 — Interactive Learning Companion

| Required Feature | How SHAPE Agent 16 delivers it |
|---|---|
| Learner profile building | SQLite `learners` table — level, topics seen, updated each turn |
| Personalised content generation | System prompt dynamically formatted with `{level}` and `{strategy}` |
| Progress assessment and feedback | Pre/post assessments + per-turn ✅/⚠️/💡 structured feedback |
| Learning path adjustment | Automatic level promotion after 3 consecutive perfect-score turns |

In [ ]:
# ── 2. Database Initialisation ────────────────────────────────────────────────
# Cherluk: I initialise the database that will hold all curriculum content.

import shape_db_handler as db
db.init_db()
print('✅ Database initialised.')
print(f'   Location: data/shape.db')

In [ ]:
# ── 3. Curriculum Design & Lesson Authoring ───────────────────────────────────
# Cherluk: I designed the lesson schema and authored the foundational curriculum.
# Each lesson has: topic, level, title, content, keywords, author.
# The keywords field is what Tool 2 (Olu's retrieval tool) searches against.

from shape_safety import validate_lesson_content

MY_LESSONS = [
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'Variables and Data Types',
        'content': (
            'A variable stores a value in memory. Python infers the type automatically.\n\n'
            'name = "Alice"   # str\n'
            'age  = 21        # int\n'
            'gpa  = 3.8       # float\n'
            'enrolled = True  # bool\n\n'
            'Practice: Create one variable of each type and print them.'
        ),
        'keywords': ['variable','variables','int','float','str','bool','type','data'],
        'author': 'Cherluk',
    },
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'For Loops and Iteration',
        'content': (
            'A for loop repeats a block of code for each item in a sequence.\n\n'
            'for i in range(5):  # prints 0 1 2 3 4\n'
            '    print(i)\n\n'
            'You can also iterate over a list:\n'
            'fruits = ["apple", "banana"]\n'
            'for fruit in fruits:\n'
            '    print(fruit)\n\n'
            'Practice: Write a loop that prints the squares of numbers 1 through 10.'
        ),
        'keywords': ['loop','loops','for','iteration','range','iterate'],
        'author': 'Cherluk',
    },
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'If / Else Decisions',
        'content': (
            'Use if/else to make your program choose between paths.\n\n'
            'score = 85\n'
            'if score >= 90:\n'
            '    grade = "A"\n'
            'elif score >= 80:\n'
            '    grade = "B"\n'
            'else:\n'
            '    grade = "C"\n\n'
            'Practice: Write an if/else that prints Pass or Fail based on a score variable.'
        ),
        'keywords': ['if','else','elif','condition','conditional','decision','branch'],
        'author': 'Cherluk',
    },
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'Functions',
        'content': (
            'A function is a reusable block of code defined with def.\n\n'
            'def greet(name):\n'
            '    return f"Hello, {name}!"\n\n'
            'print(greet("World"))  # Hello, World!\n\n'
            'Default parameters:\n'
            'def power(base, exp=2):\n'
            '    return base ** exp\n\n'
            'Practice: Write a function that returns the average of a list of numbers.'
        ),
        'keywords': ['function','functions','def','return','parameter','call','argument'],
        'author': 'Cherluk',
    },
    {
        'topic': 'AI Concepts', 'level': 'intermediate',
        'title': 'Supervised vs Unsupervised Learning',
        'content': (
            'Supervised Learning: data has labels (input→output pairs).\n'
            '  Examples: classification, regression\n'
            '  Algorithms: decision trees, SVM, neural networks\n\n'
            'Unsupervised Learning: data has NO labels.\n'
            '  Examples: clustering, dimensionality reduction\n'
            '  Algorithms: K-Means, PCA, autoencoders\n\n'
            'Key difference: presence of labelled training data.'
        ),
        'keywords': ['supervised','unsupervised','learning','classification','clustering','ml','machine'],
        'author': 'Cherluk',
    },
]

saved = 0
for lesson in MY_LESSONS:
    v = validate_lesson_content(lesson['title'], lesson['content'], lesson['keywords'])
    if not v.safe:
        print(f'❌ {lesson["title"]}: {v.reason}')
        continue
    existing = db.search_lessons(lesson['keywords'], level=lesson['level'])
    if lesson['title'] not in [e['title'] for e in existing]:
        lid = db.add_lesson(**lesson)
        print(f'✅ Saved [{lid}]: {lesson["title"]} (by {lesson["author"]})')
        saved += 1
    else:
        print(f'⏭  Already exists: {lesson["title"]}')

print(f'\n{saved} new lessons saved to curriculum database.')

In [ ]:
# ── 4. Add Your Own Lesson (Live Authoring Demo) ──────────────────────────────
# Cherluk: This demonstrates the live curriculum authoring feature I designed.
# Any team member can add a lesson here — no Python knowledge required.
# Just fill in the fields below and run the cell.

NEW_LESSON = {
    'topic':    'Python Basics',
    'level':    'beginner',
    'title':    'Lists — Creating and Accessing',
    'content':  (
        'A list stores multiple values in a single variable.\n\n'
        'fruits = ["apple", "banana", "cherry"]\n'
        'print(fruits[0])   # apple  (index starts at 0)\n'
        'print(fruits[-1])  # cherry (last element)\n\n'
        'Common list methods:\n'
        '  fruits.append("mango")   # add to end\n'
        '  fruits.remove("banana") # remove by value\n'
        '  len(fruits)              # number of items\n\n'
        'Practice: Create a list of 5 numbers and print their sum using sum().'
    ),
    'keywords': ['list','lists','append','index','remove','len','items'],
    'author':   'Cherluk',
}

v = validate_lesson_content(NEW_LESSON['title'], NEW_LESSON['content'], NEW_LESSON['keywords'])
if v.safe:
    lid = db.add_lesson(**NEW_LESSON)
    print(f'✅ Lesson "{NEW_LESSON["title"]}" saved (ID: {lid})')
    print('The agent will use this lesson immediately when learners ask about lists.')
else:
    print(f'❌ Validation error: {v.reason}')

In [ ]:
# ── 5. Verify Curriculum ──────────────────────────────────────────────────────
all_lessons = db.search_lessons(
    ['python','loop','function','variable','if','list','supervised'], level=None)
print(f'Curriculum database — {len(all_lessons)} lessons total:')
print()
for l in all_lessons:
    print(f'  [{l["level"]:12s}] {l["title"]:40s} by {l["author"]}')

---
## 6. Wake Phase — Episodic Memory Encoding  *(4-Step Memory Demo Protocol — Step 1)*

**Author: Cherluk**

As part of the 4-Step Demo Protocol I contributed **Step 1: Wake Phase**.  
My role is to show how curriculum-related facts about a learner are encoded as *episodes*  
at the start of a session, so the system can remember preferences and correct mistakes  
across sessions — not just within a single conversation.

### What is an episode?

An episode is a single memory unit with:

| Field | Meaning |
|---|---|
| `episode_type` | `fact` / `preference` / `correction` / `distractor` / `skill` |
| `subject` | Short label for what is being remembered (e.g. `preferred_language`) |
| `content` | The actual value or statement |
| `salience` | How important it is (0–1). Corrections auto-get 1.0 |
| `confidence` | `high` / `medium` / `low` |

**Corrections automatically supersede** earlier episodes on the same subject —  
so if a learner first says "I prefer Python" and later says "actually I prefer JavaScript",  
only the corrected value survives to the next session.

In [ ]:
# ── 6a. Wake Phase: encode learner facts and a mid-session correction ─────────
# Cherluk: I demonstrate how facts, preferences, distractors, and corrections
# are encoded during a live session (Step 1 of the 4-step protocol).

import uuid
from shape_memory import log_episode, get_active_episodes, get_episodes_for_session

# Simulate a session for learner 'cherluk_demo'
db.upsert_learner('cherluk_demo', 'Cherluk')
session_id = str(uuid.uuid4())
db.create_session(session_id, 'cherluk_demo')

print('WAKE PHASE — Episodic Encoding (Cherluk)')
print('='*55)
print()

# Turn 1: a persistent personal fact
e1 = log_episode(session_id, 'cherluk_demo', 1,
    episode_type='fact',
    subject='preferred_language',
    content='Python',
    confidence='high')
print(f'Turn 1 → [fact]        episode_id={e1}  subject=preferred_language  content="Python"')

# Turn 2: a learning preference
e2 = log_episode(session_id, 'cherluk_demo', 2,
    episode_type='preference',
    subject='study_schedule',
    content='mornings before 9 AM',
    confidence='medium')
print(f'Turn 2 → [preference]  episode_id={e2}  subject=study_schedule  content="mornings before 9 AM"')

# Turn 3: small-talk distractor (low salience — will be pruned in consolidation)
e3 = log_episode(session_id, 'cherluk_demo', 3,
    episode_type='distractor',
    subject='weather_comment',
    content='Learner mentioned it is raining outside.')
print(f'Turn 3 → [distractor]  episode_id={e3}  subject=weather_comment  (low salience — will be pruned)')

# Turn 4: a skill fact
e4 = log_episode(session_id, 'cherluk_demo', 4,
    episode_type='skill',
    subject='skill_for_loops',
    content='Demonstrated correct use of for loop with range() on turn 4.',
    confidence='high')
print(f'Turn 4 → [skill]       episode_id={e4}  subject=skill_for_loops')

# Turn 5: MID-SESSION CORRECTION — learner updates their language preference
e5 = log_episode(session_id, 'cherluk_demo', 5,
    episode_type='correction',
    subject='preferred_language',
    content='JavaScript — learner corrected themselves at turn 5.')
print(f'Turn 5 → [CORRECTION]  episode_id={e5}  subject=preferred_language  content="JavaScript"')
print(f'         ↳ Episode {e1} (Python) is now superseded=1 and linked via contradiction_of={e5}')

print()
print('Active (non-superseded) episodes after the session:')
active = get_active_episodes('cherluk_demo')
for ep in active:
    print(f'  id={ep["episode_id"]}  [{ep["episode_type"]:11s}]  {ep["subject"]:25s}  salience={ep["salience"]:.2f}')

print()
print(f'✅ Wake phase complete — {len(active)} active episodes ready for consolidation.')
print('   (The distractor and superseded Python episode will be pruned in Step 2.)')

---
## 7. Reflection — Cherluk

**Design decisions I made:**

- **Keyword-based schema** — I chose to tag each lesson with a `keywords` list rather than a category tree. This lets the retrieval tool (Olu's Tool 2) find lessons from natural language queries without needing a vector database.
- **Level stratification** — Separating `beginner / intermediate / advanced` means the agent never serves advanced content to a learner who hasn't demonstrated readiness, satisfying the Option 4 requirement for learning path adjustment.
- **Author attribution** — Every lesson stores the author's name. This makes it easy to see who contributed what content and supports team accountability.
- **Wake-phase episode types** — I chose 5 episode types (`fact`, `preference`, `correction`, `distractor`, `skill`) that map directly to the kinds of information the agent collects in a tutoring session. Distractors are intentionally low-salience so the consolidation job can prune them without manual intervention.

**Challenges I encountered:**

- Deciding how much detail to put in each lesson — too short and the agent has nothing to work with; too long and it overflows the LLM context window. I settled on 150–300 words per lesson.
- Ensuring keywords were specific enough to avoid false matches.
- Understanding that a `correction` episode must reference the same `subject` string as the original fact — otherwise the supersede logic won't fire. This required agreeing on subject naming conventions with the team.

**What I learned:**

- Curriculum design for AI systems is different from writing a textbook — you're writing for a retrieval index, not a human reader.
- The episodic memory layer (Step 1) is where **instructional context becomes persistent**. Without it, the agent treats every session as the first time it has met the learner.

---

## ✋ Handoff to Olu (Section 2)

**Delivered:**
- Problem statement and Option 4 alignment table
- 6 lessons in the curriculum DB (variables, loops, if/else, functions, lists, supervised learning)
- Live lesson authoring demonstration
- Wake-phase episodic encoding: 5 episodes (1 fact, 1 preference, 1 distractor, 1 skill, 1 correction) for `cherluk_demo`

**Olu needs to:** Build the tools that let the agent search this curriculum (Tool 2) and safely execute student code (Tool 1), verify the safety layer protects the episodes store, and document exactly when each tool fires.

---
*SHAPE Agent 16 — ITAI2376_14562 | Michael Rivas · **Cherluk** · Olu · Taki*